이 노트북은 GitHub 제출용 사본입니다. 원본 사진·첨부와 코드 출력은 포함하지 않습니다. 로컬 비교 그림은 PILL_EXTERNAL_EDA_DIR 환경변수로 EDA 결과 폴더를 연결하세요.

In [ ]:
import sys
from pathlib import Path

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "config.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
from notebooks.eda_report_view import load_table, report_root, show_figure

# 외부 AI Hub 알약 데이터 EDA

분석일 2026-11-05 · seed42

ZIP 라벨·목록은 전수 검사하고, 사진 픽셀은 명시한 표본만 검사했습니다. 원본 ZIP을 수정하거나 전체 압축 해제하지 않았습니다.

| 항목 | 값 |
| --- | --- |
| archives | 16 |
| archive_bytes | 29219648373 |
| images_inventory | 12009 |
| index_images | 4003 |
| json_inventory | 45934 |
| json_parse_errors | 1 |
| bbox_rows | 45933 |
| valid_bbox_rows_declared | 45709 |
| products | 116 |
| non_data_members | 22 |

## 1. 내부 데이터와 반드시 구분할 라벨 체계

`category_id`와 `categories.name`은 아래 관측값입니다. 제품 분류 키로 바로 쓰지 않습니다. `images[].drug_N` 제품 코드로 집계하고 `dl_mapping_code`·상위 폴더·제품명·품목코드의 일관성을 확인했습니다.

| 항목 | 값 |
| --- | --- |
| product_codes | 116 |
| category_id_values | ['1'] |
| category_names | ['Drug'] |

문서마다 annotation/image ID가 반복되는 것은 로컬 ID 구조입니다. `ZIP!JSON경로#행`으로 유일한 키를 만들었습니다. 외부 `dl_idx`를 내부 클래스 ID에 숫자로 바로 붙이거나 일괄 +1하지 않습니다. `tables/internal_external_product_mapping.csv`를 제품 코드·품목코드·이름으로 검수한 뒤 명시적 통합 class map을 만들어야 합니다.

## 2. 전수 라벨·목록 검사

| 항목 | 값 |
| --- | --- |
| split | {'train': {'images': 10509, 'labels': 40413, 'products': 116, 'groups': 3501}, 'valid': {'images': 1500, 'labels': 5520, 'products': 54, 'groups': 500}} |
| product_codes | 116 |
| category_id_values | ['1'] |
| category_names | ['Drug'] |
| bbox_invalid_declared | 224 |
| bbox_missing_or_non_numeric | 163 |
| bbox_nonpositive_side | 0 |
| bbox_out_of_bounds | 61 |
| area_mismatch | 623 |
| area_null_or_non_numeric | 84 |
| area_calculation_mismatch | 458 |
| image_link_unresolved | 24 |
| image_without_label | 12 |
| index_images_excluded_from_training_inventory | 4003 |
| filename_json_stem_conflicts | 24 |
| filename_missing_candidates | 29 |
| overlap_pairs | 71 |
| invalid_dates | 0 |
| catalog_conflicts | 0 |
| code_mapping_conflicts | 0 |
| code_folder_conflicts | 8 |
| train_valid_common_groups | 0 |
| train_valid_common_names | 0 |
| valid_products_absent_train | 0 |
| train_products_absent_valid | 62 |
| crc_size_candidate_images | 0 |
| metadata_rows_with_null | 4149 |
| rows_after_null_drop | 41784 |
| products_lost_after_null_drop | ['K-016548', 'K-029711', 'K-037777', 'K-038927', 'K-038954', 'K-041149', 'K-044834', 'K-053384'] |
| rows_after_blank_null_drop | 0 |
| min_boxes_per_product | 90 |
| max_boxes_per_product | 1356 |

### 사진·제품 분포

로컬 비교 그림: 사진·제품 분포 (다음 셀에서 표시)

In [ ]:
show_figure("external", "overview.png")

### 모아보기·운영체제 파일과 실제 사진을 구분

`_index.png`는 제품 코드·약 이름이 사진 안에 적힌 참고 이미지로 확인했습니다. 정답 글자가 픽셀에 포함되므로 학습 사진과 라벨 누락 집계에서 제외합니다. 예시는 아래 갤러리에 있습니다. `.DS_Store`와 `._`는 운영체제 부가 파일입니다. 진짜 JSON 파싱 실패와 구분했으며 원본은 삭제하지 않았습니다. 제공 ZIP 번호가 이어지지 않으므로 배포 전체의 완전성을 주장하지 않습니다.

### 우선 처리와 수치 해석

bbox_rows는 빈 위치 정답을 포함한 라벨 행 수입니다. bbox_invalid_declared는 bbox 결측·유효하지 않은 변·화면 밖 좌표의 합계입니다. area_mismatch도 결측 bbox/area를 포함합니다. 순수 계산 불일치는 area_calculation_mismatch입니다. 빈 bbox만 삭제하고 사진을 학습하면 해당 알약을 배경으로 배울 수 있습니다. 복구 전 사진 격리 또는 지원되는 무시 영역을 검토하세요.

| source | error |
| --- | --- |
| TL_6_조합.zip!K-012247-016551-033009-044199_json/K-016551-오류/K-012247-016551-033009-044199_0_2_0_2_90_000_200.json | JSONDecodeError: Expecting value: line 59 column 12 (char 1571) |

## 3. 제품별 분포

| product_code | pill_name | boxes | photos | groups | splits | dl_idx | item_seq |
| --- | --- | --- | --- | --- | --- | --- | --- |
| K-016548 | 가바토파정 100mg | 1356 | 1353 | 452 | train,valid | 16547 | 200500251 |
| K-016551 | 동아가바펜틴정 800mg | 1340 | 1339 | 447 | train,valid | 16550 | 200500254 |
| K-016232 | 리피토정 20mg | 807 | 807 | 269 | train | 16231 | 200410085 |
| K-016262 | 크레스토정 20mg | 798 | 798 | 266 | train,valid | 16261 | 200410326 |
| K-003614 | 동아오팔몬정(리마프로스트알파-시클로덱스트린포접화합물) | 642 | 642 | 214 | train | 3613 | 199200261 |
| K-011220 | 아스피린프로텍트정 100mg | 642 | 642 | 214 | train | 11219 | 200108429 |
| K-015710 | 엔테론정 150mg | 639 | 639 | 213 | train | 15709 | 200403500 |
| K-016235 | 카듀엣정 5mg/20mg | 639 | 639 | 213 | train,valid | 16234 | 200410088 |
| K-020238 | 플라빅스정 75mg | 621 | 621 | 207 | train,valid | 20237 | 200700907 |
| K-029667 | 리바로정 4mg | 612 | 612 | 204 | train,valid | 29666 | 201205253 |
| K-036637 | 로수젯정10/5밀리그램 | 612 | 612 | 204 | train,valid | 36636 | 201503658 |
| K-035206 | 아토젯정 10/40mg | 600 | 600 | 200 | train,valid | 35205 | 201500365 |
| K-013004 | 이가탄에프캡슐 150mg/PTP | 540 | 540 | 180 | train | 13003 | 200300913 |
| K-000250 | 마그밀정(수산화마그네슘) | 540 | 540 | 180 | train | 249 | 197400246 |
| K-010487 | 동화덴스톨캡슐 150mg/PTP | 540 | 540 | 180 | train | 10486 | 200100780 |
| K-006835 | 이튼큐정 35mg/PTP | 539 | 539 | 180 | train | 6834 | 199704687 |
| K-012638 | 투스딘골드캡슐 150mg/PTP | 537 | 537 | 179 | train | 12637 | 200300092 |
| K-001866 | 알마겔정(알마게이트)(수출명:유한가스트라겔정) | 537 | 537 | 179 | train | 1865 | 198700430 |
| K-020877 | 엑스포지정 5/160mg | 468 | 468 | 156 | train,valid | 20876 | 200703803 |
| K-027653 | 세비카정 10/40mg | 468 | 468 | 156 | train,valid | 27652 | 201004570 |
| K-027733 | 트윈스타정 40/5mg | 468 | 468 | 156 | train,valid | 27732 | 201005081 |
| K-022347 | 자누비아정 50mg | 468 | 468 | 156 | train,valid | 22346 | 200710758 |
| K-027777 | 카나브정 60mg | 467 | 467 | 156 | train,valid | 27776 | 201005501 |
| K-003743 | 알드린정 | 465 | 465 | 155 | train | 3742 | 199201106 |
| K-034597 | 제미메트서방정 50/1000mg | 465 | 465 | 155 | train,valid | 34596 | 201405223 |
| K-025367 | 자누메트정 50/850mg | 465 | 465 | 155 | train,valid | 25366 | 200901817 |
| K-030308 | 트라젠타듀오정 2.5/850mg | 462 | 462 | 154 | train,valid | 30307 | 201208291 |
| K-028763 | 트라젠타정(리나글립틴) | 462 | 462 | 154 | train,valid | 28762 | 201106367 |
| K-025469 | 아모잘탄정 5/100mg | 459 | 459 | 153 | train,valid | 25468 | 200902349 |
| K-031885 | 자누메트엑스알서방정 100/1000mg | 459 | 459 | 153 | train,valid | 31884 | 201309418 |

전체 116행 중 30행 표시. tables 폴더에 전체 행 저장.

전체 제품은 tables/product_distribution.csv에 있습니다. 반복 촬영 횟수와 제품 수를 구분하고, 클래스 불균형 보정은 train에서만 비교하세요.

## 4. 결측 피처·의미·처리

| section | field | rows | key_missing | null | empty_string | whitespace | text_missing | empty_container | present |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| images | change_date | 45933 | 0 | 4149 | 0 | 0 | 0 | 0 | 41784 |
| images | chart | 45933 | 0 | 1356 | 0 | 0 | 0 | 0 | 44577 |
| images | color_class1 | 45933 | 0 | 1356 | 0 | 0 | 0 | 0 | 44577 |
| images | color_class2 | 45933 | 0 | 1356 | 37506 | 0 | 0 | 0 | 7071 |
| images | di_company_mf | 45933 | 0 | 0 | 34064 | 0 | 0 | 0 | 11869 |
| images | di_company_mf_en | 45933 | 0 | 0 | 38957 | 0 | 0 | 0 | 6976 |
| images | dl_material | 45933 | 0 | 0 | 1323 | 0 | 0 | 0 | 44610 |
| images | dl_material_en | 45933 | 0 | 0 | 1206 | 0 | 0 | 0 | 44727 |
| images | dl_name_en | 45933 | 0 | 0 | 159 | 0 | 0 | 0 | 45774 |
| images | drug_shape | 45933 | 0 | 1356 | 0 | 0 | 0 | 0 | 44577 |
| images | form_code_name | 45933 | 0 | 1356 | 0 | 0 | 0 | 0 | 44577 |
| images | img_key | 45933 | 0 | 1356 | 0 | 0 | 0 | 0 | 44577 |
| images | img_regist_ts | 45933 | 0 | 1356 | 0 | 0 | 0 | 0 | 44577 |
| images | line_back | 45933 | 0 | 1356 | 40728 | 0 | 0 | 0 | 3849 |
| images | line_front | 45933 | 0 | 1356 | 39201 | 0 | 0 | 0 | 5376 |
| images | mark_code_back | 45933 | 0 | 1356 | 44577 | 0 | 0 | 0 | 0 |
| images | mark_code_back_anal | 45933 | 0 | 1356 | 44037 | 0 | 0 | 0 | 540 |
| images | mark_code_back_img | 45933 | 0 | 1356 | 44037 | 0 | 0 | 0 | 540 |
| images | mark_code_front | 45933 | 0 | 1356 | 44577 | 0 | 0 | 0 | 0 |
| images | mark_code_front_anal | 45933 | 0 | 1356 | 35604 | 0 | 0 | 0 | 8973 |
| images | mark_code_front_img | 45933 | 0 | 1356 | 35604 | 0 | 0 | 0 | 8973 |
| images | print_back | 45933 | 0 | 1356 | 25467 | 0 | 0 | 0 | 19110 |
| images | print_front | 45933 | 0 | 1356 | 360 | 0 | 0 | 0 | 44217 |

null·빈 문자열·키 부재를 따로 집계했습니다. 빈 각인·색·마크는 실제 없음으로 확정할 수 없습니다. 제품별 전체·부분 결측은 missing_by_product.csv에서 구분합니다.

| 피처 | 의미와 처리 근거 |
|---|---|
| print_front/back | 조건부 설명. null·빈 값을 무각인 OCR 정답으로 바꾸지 않음 |
| color_class2·mark | 미기재와 실제 단색·무로고 구분. 다른 제품 최빈값 복사 금지 |
| 물리 치수 0·결측 | 분석용 NA+invalid. 카메라0·ignore0은 유효 값으로 보존 |
| segmentation=[] | bbox 전용 라벨의 마스크 미제공. bbox 결측과 구분 |
| 설명 전체 미기재 열 | 모델 입력에서 제외 가능하지만 사진·bbox·원문은 보존 |

결측 원인은 수집·연계 누락이나 선택 필드 미기재일 수 있습니다. 원문만으로 확정하지 않습니다. 정답 제품 설명을 이미지 입력에 붙이면 정답 누수가 생길 수 있습니다. 색·모양 보조 정답이 없으면 해당 loss만 제외합니다. OCR은 사진에서 읽은 결과를 전체 후보 카탈로그와 비교합니다.

### 피처별 결측

로컬 비교 그림: 피처별 결측 (다음 셀에서 표시)

In [ ]:
show_figure("external", "missing.png")

## 5. bbox·면적·치수·누락 후보

| product_code | pill_name | bbox | area | bbox_area | area_delta | bbox_valid_declared |
| --- | --- | --- | --- | --- | --- | --- |
| K-012778 | 다보타민큐정 10mg/병 | [] | None | nan | nan | False |
| K-000573 | 게보린정 300mg/PTP | [] | None | nan | nan | False |
| K-023223 | 비타비백정 100mg/병 | [] | None | nan | nan | False |
| K-037777 | 알바스테인캡슐(에르도스테인) | [] | None | nan | nan | False |
| K-000250 | 마그밀정(수산화마그네슘) | [] | None | nan | nan | False |
| K-002483 | 뮤테란캡슐 100mg | [] | None | nan | nan | False |
| K-000250 | 마그밀정(수산화마그네슘) | [] | None | nan | nan | False |
| K-000250 | 마그밀정(수산화마그네슘) | [] | None | nan | nan | False |
| K-002483 | 뮤테란캡슐 100mg | [] | None | nan | nan | False |
| K-002483 | 뮤테란캡슐 100mg | [93, 674, 210, 488] | 10248 | 102480.0 | -92232.0 | True |
| K-022362 | 맥시부펜이알정 300mg | [145, 855, 284, 299] | 108204 | 84916.0 | 23288.0 | True |
| K-022627 | 메가파워정 90mg/병 | [121125, 333, 475] | 158175 | nan | nan | False |
| K-000250 | 마그밀정(수산화마그네슘) | [121, 802, 279, 283] | 48957 | 78957.0 | -30000.0 | True |
| K-000250 | 마그밀정(수산화마그네슘) | [128.816, 287, 299] | 85813 | nan | nan | False |
| K-000250 | 마그밀정(수산화마그네슘) | [123.816, 289, 298] | 86122 | nan | nan | False |
| K-004378 | 타이레놀정500mg | [133, 107, 218, 40] | 95920 | 8720.0 | 87200.0 | True |
| K-019552 | 트루비타정 60mg/병 | [523, 637, 35, 475] | 163875 | 16625.0 | 147250.0 | True |
| K-004378 | 타이레놀정500mg | [135, 110, 219, 435] | 95625 | 95265.0 | 360.0 | True |
| K-037777 | 알바스테인캡슐(에르도스테인) | [644, 851, 190, 498] | 94620 | 94620.0 | 0.0 | False |
| K-000250 | 마그밀정(수산화마그네슘) | [121, 802, 278, 284] | 48952 | 78952.0 | -30000.0 | True |
| K-000250 | 마그밀정(수산화마그네슘) | [115, 810, 284, 295] | 89780 | 83780.0 | 6000.0 | True |
| K-005002 | 엘도스캡슐(에르도스테인)(수출용) | [618, 97, 210, 494] | 103470 | 103740.0 | -270.0 | True |
| K-012081 | 리렉스펜정 300mg/PTP | [652, 705239, 472] | 112808 | nan | nan | False |
| K-005002 | 엘도스캡슐(에르도스테인)(수출용) | [622.9, 209, 485] | 101365 | nan | nan | False |
| K-005002 | 엘도스캡슐(에르도스테인)(수출용) | [617.141, 207, 471] | 97497 | nan | nan | False |
| K-005002 | 엘도스캡슐(에르도스테인)(수출용) | [632, 98, 131, 497] | 94927 | 65107.0 | 29820.0 | True |
| K-000250 | 마그밀정(수산화마그네슘) | [812, 185, 272, 275] | 74800 | 74800.0 | 0.0 | False |
| K-013161 | 엘스테인캡슐(에르도스테인) | [624, 73, 227, 485] | 110005 | 110095.0 | -90.0 | True |
| K-006192 | 삐콤씨에프정 618.6mg/병 | [550, 557, 269, 539] | 144184 | 144991.0 | -807.0 | True |
| K-044834 | 엘스테인정(에르도스테인) | [605, 219, 264, 267] | 70288 | 70488.0 | -200.0 | True |

전체 657행 중 30행 표시. tables 폴더에 전체 행 저장.

| product_code | field | value | nonpositive_rows | rows | reason | lower | upper |
| --- | --- | --- | --- | --- | --- | --- | --- |
| K-010487 | thick | 0.0 | 540 | 540 | 물리 치수 0·결측 | -0.3149999999999995 | 10.785 |
| K-010487 | leng_long | 0.0 | 540 | 540 | 물리 치수 0·결측 | -5.968749999999998 | 32.68125 |
| K-010487 | leng_short | 0.0 | 540 | 540 | 물리 치수 0·결측 | 2.6250000000000004 | 12.825 |
| K-016548 | thick | 0.0 | 1356 | 1356 | 물리 치수 0·결측 | -0.3149999999999995 | 10.785 |
| K-016548 | leng_long | 0.0 | 1356 | 1356 | 물리 치수 0·결측 | -5.968749999999998 | 32.68125 |
| K-016548 | leng_short | 0.0 | 1356 | 1356 | 물리 치수 0·결측 | 2.6250000000000004 | 12.825 |
| K-026993 | leng_short | 14.8 | 0 | 450 | 제품별 대표값 IQR 후보 · 오류 확정 아님 | 2.6250000000000004 | 12.825 |

IQR은 양수인 제품별 대표값을 한 번씩 사용했습니다. 후보는 오류 확정이나 삭제 지시가 아닙니다. 좌표 검사는 원문 width/height 기준이며 실제 픽셀 크기 대조는 표본에 한정됩니다. 면적 오류는 bbox 자체를 먼저 검수한 뒤 최종 w×h를 계산합니다. 범위 내 박스도 몸통 잘림·다른 알약 라벨일 수 있습니다. 수치 통과를 정상 인증으로 쓰지 않습니다.

| split | file_name | expected_code | reason |
| --- | --- | --- | --- |
| train | K-001900-003544-004543-016548_0_2_0_2_90_000_200.png | K-016548 | 파일명 기대 코드에 연결된 JSON 없음 · 시각 확정 전 |
| train | K-001900-003544-004543-016551_0_2_0_2_90_000_200.png | K-016551 | 파일명 기대 코드에 연결된 JSON 없음 · 시각 확정 전 |
| train | K-001900-003544-006563-016551_0_2_0_2_90_000_200.png | K-001900 | 파일명 기대 코드에 연결된 JSON 없음 · 시각 확정 전 |
| train | K-001900-003544-016548-021026_0_2_0_2_70_000_200.png | K-021026 | 파일명 기대 코드에 연결된 JSON 없음 · 시각 확정 전 |
| train | K-001900-003544-016548-021026_0_2_0_2_90_000_200.png | K-021026 | 파일명 기대 코드에 연결된 JSON 없음 · 시각 확정 전 |
| train | K-001900-003544-016548-027926_0_2_0_2_90_000_200.png | K-027926 | 파일명 기대 코드에 연결된 JSON 없음 · 시각 확정 전 |
| train | K-001900-003544-016548-029345_0_2_0_2_70_000_200.png | K-016548 | 파일명 기대 코드에 연결된 JSON 없음 · 시각 확정 전 |
| train | K-001900-003544-016548-029345_0_2_0_2_90_000_200.png | K-029345 | 파일명 기대 코드에 연결된 JSON 없음 · 시각 확정 전 |
| train | K-001900-003544-016548-031705_0_2_0_2_90_000_200.png | K-001900 | 파일명 기대 코드에 연결된 JSON 없음 · 시각 확정 전 |
| train | K-001900-003544-016548-033009_0_2_0_2_90_000_200.png | K-033009 | 파일명 기대 코드에 연결된 JSON 없음 · 시각 확정 전 |
| train | K-001900-003544-016551-029451_0_2_0_2_70_000_200.png | K-001900 | 파일명 기대 코드에 연결된 JSON 없음 · 시각 확정 전 |
| train | K-001900-003544-016551-029451_0_2_0_2_75_000_200.png | K-029451 | 파일명 기대 코드에 연결된 JSON 없음 · 시각 확정 전 |
| train | K-001900-003544-016551-031705_0_2_0_2_75_000_200.png | K-031705 | 파일명 기대 코드에 연결된 JSON 없음 · 시각 확정 전 |
| train | K-001900-003544-016551-031705_0_2_0_2_90_000_200.png | K-031705 | 파일명 기대 코드에 연결된 JSON 없음 · 시각 확정 전 |
| train | K-001900-004543-016548-044199_0_2_0_2_70_000_200.png | K-001900 | 파일명 기대 코드에 연결된 JSON 없음 · 시각 확정 전 |

전체 29행 중 15행 표시. tables 폴더에 전체 행 저장.

파일명 기대 코드가 없으면 라벨 연결 누락 후보입니다. 파일명만으로 정답을 생성하지 않습니다. 미라벨 물체가 실제로 있으면 복구 전 사진 제외 또는 지원되는 무시 영역을 검토합니다.

### bbox 분포

로컬 비교 그림: bbox 분포 (다음 셀에서 표시)

In [ ]:
show_figure("external", "bbox.png")

## 6. 분할과 내부 데이터 중복

| 항목 | 값 |
| --- | --- |
| checked | True |
| shared_product_codes | 54 |
| same_name_image_candidates | 0 |
| matched_product_dlidx_differences | 54 |
| matched_product_itemseq_conflicts | 0 |

같은 이름의 내부·외부 사진이 있을 경우 SHA256을 대조합니다. 동일 이름이 없어도 다른 이름의 중복까지 없다고 해석하지 않습니다. 내부 train과 동일한 외부 사진을 검증에 두지 마세요. 같은 촬영 조합의 각도·방향 변형도 그룹으로 묶고 클래스 대표성을 확인합니다. 공통 촬영 조합과 train에 없는 valid 제품도 표로 남겼습니다. CRC32+크기는 중복 후보용입니다. 정확 중복 확정은 읽은 이미지 SHA256에만 적용합니다.

## 7. 이미지 검사 범위

| 항목 | 값 |
| --- | --- |
| decoded_images_attempted | 159 |
| random_sample_images | 96 |
| decode_errors_in_sample | 0 |
| index_images_decoded | 26 |
| photographs_decoded | 133 |
| photographs_not_decoded | 11876 |
| sample_dimensions | {(976, 1280): 159} |

전체 이미지 손상 없음·전체 중복 없음으로 일반화하지 않습니다. 무작위 표본·문제 후보·비교 이미지·내부와 같은 이름의 사진을 실제 디코딩했습니다. 밝기·대비는 장면 전체의 기술 지표이며 불량 사진 판정이 아닙니다. ZIP 전체 testzip·사진 전수 SHA256·OCR·오라벨 전수 검수·모델 학습은 생략했습니다.

## 8. 가로 비교 이미지

같은 제품·문제 유형끼리 한 줄에 최대 세 장을 배치했습니다. 제품 코드·약 이름·bbox 좌표·각인·치수와 검토 이유는 원문 기준입니다.

### 수치 검사로 잡히지 않는 같은 클래스의 외형·각인 차이

실제 사진을 확인한 세 사례입니다. 현재 라벨은 모두 K-016548/가바토파이지만 사진에는 GB, Airtal, SK가 관측됩니다. 특히 Airtal 사례는 GB 참조와 색·외형도 다릅니다. 같은 제품의 관측면 차이인지, 박스 대상이나 제품 연결이 잘못됐는지 원본·카탈로그로 검수해야 합니다. 제품 정체·새 클래스 번호를 자동 확정하거나 원본을 수정하지 않았습니다. 전체 라벨에 대한 의미상 전수 검수 결과는 아닙니다.

| product_code | observed_text | review_note |
| --- | --- | --- |
| K-016548 | GB | 같은 라벨의 촬영 참조 · 실제 제품 정체 보증 아님 |
| K-016548 | Airtal | GB 참조와 각인·색·외형이 다름 · 라벨 연결 검수 후보 |
| K-016548 | SK | 같은 제품 코드지만 각인·외형이 다름 · 관측면도 대조 필요 |

로컬 비교 그림: 같은 클래스의 외형 검수 (다음 셀에서 표시)

In [ ]:
show_figure("external", "semantic_review_01.jpg")

### bbox 빈 리스트·수치 결측 · 같은 제품의 다른 촬영

로컬 비교 그림: bbox 빈 리스트·수치 결측 · 같은 제품의 다른 촬영 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_01.jpg")

### bbox 빈 리스트·수치 결측 · 같은 제품의 다른 촬영

로컬 비교 그림: bbox 빈 리스트·수치 결측 · 같은 제품의 다른 촬영 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_02.jpg")

### 화면 범위를 벗어난 bbox

로컬 비교 그림: 화면 범위를 벗어난 bbox (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_03.jpg")

### 화면 범위를 벗어난 bbox

로컬 비교 그림: 화면 범위를 벗어난 bbox (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_04.jpg")

### 화면 범위를 벗어난 bbox

로컬 비교 그림: 화면 범위를 벗어난 bbox (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_05.jpg")

### 같은 제품 · bbox 면적 불일치

로컬 비교 그림: 같은 제품 · bbox 면적 불일치 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_06.jpg")

### 같은 제품 · bbox 면적 불일치

로컬 비교 그림: 같은 제품 · bbox 면적 불일치 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_07.jpg")

### 같은 제품 · bbox 면적 불일치

로컬 비교 그림: 같은 제품 · bbox 면적 불일치 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_08.jpg")

### 같은 제품 · bbox 면적 불일치

로컬 비교 그림: 같은 제품 · bbox 면적 불일치 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_09.jpg")

### 같은 제품 · bbox 면적 불일치

로컬 비교 그림: 같은 제품 · bbox 면적 불일치 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_10.jpg")

### 서로 다른 라벨의 같은 위치 · 충돌 후보

로컬 비교 그림: 서로 다른 라벨의 같은 위치 · 충돌 후보 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_11.jpg")

### 서로 다른 라벨의 같은 위치 · 충돌 후보

로컬 비교 그림: 서로 다른 라벨의 같은 위치 · 충돌 후보 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_12.jpg")

### 서로 다른 라벨의 같은 위치 · 충돌 후보

로컬 비교 그림: 서로 다른 라벨의 같은 위치 · 충돌 후보 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_13.jpg")

### 설명 결측 · 사진과 bbox는 존재

로컬 비교 그림: 설명 결측 · 사진과 bbox는 존재 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_14.jpg")

### 앞 각인 빈 문자열 · 같은 제품의 다른 각도

로컬 비교 그림: 앞 각인 빈 문자열 · 같은 제품의 다른 각도 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_15.jpg")

### 앞 각인 빈 문자열 · 같은 제품의 다른 각도

로컬 비교 그림: 앞 각인 빈 문자열 · 같은 제품의 다른 각도 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_16.jpg")

### 두 가지 색 설명 · 같은 제품의 다른 촬영

로컬 비교 그림: 두 가지 색 설명 · 같은 제품의 다른 촬영 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_17.jpg")

### 두 가지 색 설명 · 같은 제품의 다른 촬영

로컬 비교 그림: 두 가지 색 설명 · 같은 제품의 다른 촬영 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_18.jpg")

### 물리 치수 IQR 후보 · 크기만으로 삭제 금지

로컬 비교 그림: 물리 치수 IQR 후보 · 크기만으로 삭제 금지 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_19.jpg")

### train 대표 제품 · 클래스 코드·약 이름·bbox 비교

로컬 비교 그림: train 대표 제품 · 클래스 코드·약 이름·bbox 비교 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_20.jpg")

### valid 대표 제품 · 클래스 코드·약 이름·bbox 비교

로컬 비교 그림: valid 대표 제품 · 클래스 코드·약 이름·bbox 비교 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_21.jpg")

### valid 대표 제품 · 클래스 코드·약 이름·bbox 비교

로컬 비교 그림: valid 대표 제품 · 클래스 코드·약 이름·bbox 비교 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_22.jpg")

### _index.png 모아보기 · 학습 사진과 분리

로컬 비교 그림: _index.png 모아보기 · 학습 사진과 분리 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_23.jpg")

### 실제 사진에 연결 JSON이 없는 후보

로컬 비교 그림: 실제 사진에 연결 JSON이 없는 후보 (다음 셀에서 표시)

In [ ]:
show_figure("external", "gallery_24.jpg")

## 9. 재현과 한계

전체 CSV와 출처는 tables 폴더에 있습니다. ZIP 목록 fingerprint와 라벨 SHA256, 표본 사진 SHA256을 보존했습니다. ZIP 전체 바이트 해시는 계산하지 않았습니다. 번호가 이어지지 않는 ZIP 구성이므로 AI Hub 배포 전체의 완전성을 주장하지 않습니다.

현재 폴더에서 재생성 (입력 경로는 환경변수로 지정):

```powershell
uv run python analyze_external.py --data-dir $env:PILL_EXTERNAL_DATA_DIR --internal-dir ../../datasets/internal_raw --output-dir .
```

In [ ]:
if (report_root("external") / "tables" / "annotations.csv").is_file():
    bbox_df = load_table("external", "annotations.csv")
    display(bbox_df.head().T)
else:
    print("PILL_EXTERNAL_EDA_DIR을 로컬 EDA 결과 폴더로 설정하세요.")